# Step 4. 4대 영역별 다변량 파생 피쳐 생성
> 거래량 과열, 가격 변동성, 캔들 마이크로 구조, 시계열 지속성을 반영한 다변량 이상 탐지 전용 피쳐 매트릭스 빌드


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(".")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

def build_anomaly_features_pipeline(file_path, output_name):
    df = pd.read_csv(file_path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).copy()
    
    # ── 영역 1. 거래량 과열 지표 ──
    df["prev_volume"] = df.groupby("ticker")["volume"].shift(1)
    df["vol_chg_rate"] = (df["volume"] - df["prev_volume"]) / df["prev_volume"]
    df["volume_ma20_ratio"] = df["volume"] / df.groupby("ticker")["volume"].transform(lambda x: x.rolling(20).mean())
    
    # ── 영역 2. 가격 변동성 지표 ──
    df["prev_close"] = df.groupby("ticker")["close"].shift(1)
    df["daily_return"] = (df["close"] - df["prev_close"]) / df["prev_close"]
    df["volatility_5d"] = df.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(5).std())
    rolling_high_5d = df.groupby("ticker")["high"].transform(lambda x: x.rolling(5).max())
    df["drawdown_after_peak_5d"] = (rolling_high_5d - df["close"]) / rolling_high_5d
    
    # ── 영역 3. 캔들 마이크로 구조 지표 ──
    candle_range = (df["high"] - df["low"]).replace(0, np.nan)
    df["upper_shadow_ratio"] = (df["high"] - df[["open", "close"]].max(axis=1)) / candle_range
    df["body_ratio"] = (df["close"] - df["open"]).abs() / candle_range
    
    # ── 영역 4. 시계열 지속성 지표 ──
    df["is_upper_shadow"] = (df["upper_shadow_ratio"] > 0.4).astype(int)
    df["upper_shadow_streak_5d"] = df.groupby("ticker")["is_upper_shadow"].transform(lambda x: x.rolling(5).sum())
    
    # 결측치 최종 제거 및 저장
    essential_features = ["volume_ma20_ratio", "volatility_5d", "upper_shadow_streak_5d"]
    df.dropna(subset=essential_features, inplace=True)
    df.drop(columns=["prev_volume", "prev_close", "is_upper_shadow"], errors="ignore", inplace=True)
    
    df.to_csv(DATA_PROCESSED / f"{output_name}_features.csv", index=False, encoding="utf-8-sig")
    print(f"💾 [{output_name.upper()}] 4대 핵심 영역별 마스터 피쳐 적재 완료 -> 구조: {df.shape}")
    return df

train_features = build_anomaly_features_pipeline(DATA_RAW / "train.csv", "train")
valid_features = build_anomaly_features_pipeline(DATA_RAW / "valid.csv", "valid")
test_features  = build_anomaly_features_pipeline(DATA_RAW / "test.csv", "test")
